# Convert Student Paper Index spreadsheet to Invenio json.

In [8]:
import pandas as pd
import json
import os

In [9]:
dir = '/Users/metadatagamechanger/Repositories/MooreaStudentPapers/'
input = dir + 'data/index_to_moorea_class_projects_1992-2022.csv'
input_df = pd.read_csv(input)
input_df.head()

,"Authors, Primary",Title Primary,organism,where,Pub Year,Keywords,Abstract,Link
0,"Alcalay, Madison","Distribution, thermal tolerance, and feeding b...",NaN,NaN,2019,NaN,NaN,"Distribution, thermal tolerance, and feeding b..."
1,"Bellegarda, Celine",Effects of thermal variability on the photosyn...,NaN,NaN,2019,NaN,NaN,Effects of thermal variability on the photosyn...
2,"Cervantes-Hernandez, Daisy",Tropical stream habitat structure and freshwat...,NaN,NaN,2019,NaN,NaN,Tropical stream habitat structure and freshwat...
3,"Glasmann, Chrissy",Lunar cycle and abundance of Copula sivickisi;...,NaN,NaN,2019,NaN,NaN,Lunar cycle and abundance of Copula sivickisi;...
4,"Golde, Chloe",The effect of rising ocean temperatures on Tri...,NaN,NaN,2019,NaN,NaN,The effect of rising ocean temperatures on Tri...


In [10]:
def rowToMetadata(i: int,           # row index in dataframe
                  papers_df,
                  paperIndex: int,  # index of paper in publication year
                  type:       str   # type of metaata (invenio or dataCite)
                  ):

    if type == 'invenio':
        #    
        # initialize invenio dictionary
        # 
        metadata_d = {
        "access": {
            "files": "public",
            "record": "public"
        },
        "files": {
            "enabled": True
        },
        "metadata": {}
        }
        metadata_d['metadata']['creators'] = []              # Add creators to metadata dictionary, initialize as empty list

        for p, person in enumerate(input_df.at[i, 'Authors, Primary'].split(';')):
            person = person.replace('Dr. ','')
            if ',' in person:
                toks = person.split(',')
            else:
                toks = person.split(' ,')

            if len(toks) < 2 or len(toks) > 4:
                continue

            family = toks[0].replace(',','') if ',' in person else toks[1]
            family = family.strip()

            given = " ".join(toks[1:]) if ',' in person else toks[0]
            given = given.strip()                                       # .replace(' ','+')

            metadata_d['metadata']['creators'].append({'person_or_org': {'family_name': family, 'given_name': given}, 'type': 'personal'})

        if len(str(input_df.at[i, 'Abstract'])) > 0 and str(input_df.at[i, 'Abstract']) != 'nan':
            metadata_d['metadata']['description'] = input_df.at[i, 'Abstract']
        #
        # Create a unique identifier for each record using a combination of the iPlaces DOI prefix, the abbreviation for the class
        # (Biology and Geology of Tropical Islands), the publication year, the paper index, and creator names.
        # The creator names are concatenated together with underscores, and spaces and special characters are removed or 
        # replaced to ensure the identifier is valid.
        #
        identifier = f"{'_'.join([ x['person_or_org']['family_name'].replace(' ','')+'_'+ x['person_or_org']['given_name'].replace(' ','').replace('.','') for x in metadata_d['metadata']['creators'] ])}"
        identifier = identifier.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
        identifier = f"10.17913/bgtl/{str(input_df.at[i, 'Pub Year'])}/{paperIndex}/{identifier}"
        metadata_d['metadata']['identifiers'] = [{'identifier': identifier, 'scheme': 'doi'}]

        metadata_d['metadata']['publication_date']   = input_df.at[i, 'Pub Year']
        metadata_d['metadata']['publisher']          = 'University of California, Berkeley'
        metadata_d['metadata']['resource_type']      = 'publication-preprint'

        contributor_Gump = {                                        # define Gump as a contributor
            "person_or_org": {
                "name": "Gump South Pacific Research Station",
                "type": "organizational",
                "identifiers": [{
                "scheme": "ror",
                "identifier": "04sk0et52"
                }],
            },
            "role": {"id": "Sponsor"},
            "affiliations": [{
                "id": "04sk0et52",
                "name": "Gump South Pacific Research Station",
            }]
        }
        metadata_d['metadata']['contributors']      = [contributor_Gump]

        if len(str(input_df.at[i, 'Keywords'])) > 0 and str(input_df.at[i, 'Keywords']) != 'nan':
            keywords_l = str(input_df.at[i, 'Keywords']).split(';')              # some keywords are delimited with ";"
            if len(keywords_l) == 1:
                keywords_l = str(input_df.at[i, 'Keywords']).split(',')          # some keywords are delimited with ","

            for subject in keywords_l:
                subject = subject.strip()
                if len(subject) > 0:
                    if 'subjects' not in metadata_d['metadata']:
                        metadata_d['metadata']['subjects'] = []
                    metadata_d['metadata']['subjects'].append({'subject': subject})
        #
        # The index spreadsheet includes a column named "organism" that includes fairly high-level organism keywords
        # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
        # In this case that is set to "organism" to differentiate these keywords from the general set
        #  
        if all(['organism' in input_df.columns, 
                len(str(input_df.at[i, 'organism'])),
                str(input_df.at[i, 'organism']) != 'nan']):
            if 'subjects' not in metadata_d['metadata']:
                metadata_d['metadata']['subjects'] = []
            metadata_d['metadata']['subjects'].append({'subject': input_df.at[i, 'organism'], 'subjectScheme':'organism'})

        #
        # The index spreadsheet includes a column named "where" that includes fairly high-level location keywords
        # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
        # In this case that is set to "where" to differentiate these keywords from the general set.
        #  
        if all(['where' in input_df.columns, 
                len(str(input_df.at[i, 'where'])),
                str(input_df.at[i, 'where']) != 'nan']):
            if 'subjects' not in metadata_d['metadata']:
                metadata_d['metadata']['subjects'] = []
            metadata_d['metadata']['subjects'].append({'subject': input_df.at[i, 'where'], 'subjectScheme':'where'})

        if len(str(input_df.at[i, 'Title Primary'])) > 0 and str(input_df.at[i, 'Title Primary']) != 'nan':
            metadata_d['metadata']['title'] = input_df.at[i, 'Title Primary']

        relatedIsPartOf = f"10.17913/bgtl/{str(input_df.at[i, 'Pub Year'])}"
        metadata_d['metadata']['related_identifiers'] = [{'identifier': relatedIsPartOf, 
                                                        'relation_type': {'id': 'ispartof'},
                                                        'resource_type': {'id': 'other'},
                                                        'scheme': 'doi'}]

        metadata_d['metadata']['version'] = 1

        fileName = f"{'_'.join([ x['person_or_org']['family_name'].replace(' ','')+'_'+ x['person_or_org']['given_name'].replace(' ','').replace('.','') for x in metadata_d['metadata']['creators'] ])}.json"
        fileName = fileName.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
        fileName = f'{str(metadata_d["metadata"]["publication_date"])}/' + fileName
        
    elif type == 'dataCite':
        pass

    return metadata_d, fileName


## Create Invenio metadata records.
See https://inveniordm.docs.cern.ch/reference/metadata/ for reference

In [11]:
currentYear = 0                                             # initialize publication year variable, will be updated for each record in loop below
paperIndex  = 0                                             # initialize paper index variable, will be updated for each record in loop below
for i in input_df.index:                                    # loop input rows (one paper/metadata record per row)

    pubYear = input_df.at[i, 'Pub Year']                # update publication year for current record
    if pubYear != currentYear:                          # new year, initialize paperIndex
        paperIndex = 1
        currentYear = pubYear
    else:                                               # same year as previous record, increment paperIndex
        paperIndex += 1

    metadata_d, fileName = rowToMetadata(i, input_df, paperIndex, type='invenio')     # create metadata dictionary for current record
    output = f'{dir}metadata/{fileName}'
    
    os.makedirs(os.path.dirname(output), exist_ok=True)
    print(f"writing metadata to {'/'.join(output.split('/')[-2:])}")
    with open(output, 'w') as f:
        json.dump(metadata_d, f, indent=2, default=str)


writing metadata to 2019/Alcalay_Madison.json
writing metadata to 2019/Bellegarda_Celine.json
writing metadata to 2019/Cervantes-Hernandez_Daisy.json
writing metadata to 2019/Glasmann_Chrissy.json
writing metadata to 2019/Golde_Chloe.json
writing metadata to 2019/Grove_Kyra.json
writing metadata to 2019/Hallsten_Molly.json
writing metadata to 2019/Kampman_Lucas.json
writing metadata to 2019/LevequeEichhorn_Lily.json
writing metadata to 2019/Lopez_Giselle.json
writing metadata to 2019/McCarron_Christopher.json
writing metadata to 2019/Petrocelly_Gabby.json
writing metadata to 2019/Rocheleau_Lauren.json
writing metadata to 2019/Schwartz_Kyle.json
writing metadata to 2019/Sezginer_Yayla.json
writing metadata to 2019/Shein_Ben.json
writing metadata to 2019/Soriano_JasonMark.json
writing metadata to 2019/Stoner-Osborne_Blake.json
writing metadata to 2019/Sturla_Savannah.json
writing metadata to 2019/Zane_Lauren.json
writing metadata to 2019/Zhang_Lian.json
writing metadata to 2018/BOVILLE_E